# GeoMarketing IDF : J15b — Des libellés français et des unités explicites dans le catalogue

## Objectifs

- Rendre les noms de variables compréhensibles pour un public non technique.
- Distinguer habitants, personnes, ménages, familles, emplois, observations, capacités et surfaces.
- Expliciter pourcentages, points de pourcentage, ratios et indices.
- Conserver les identifiants techniques, les valeurs, les sources et les géographies.
- Actualiser ensemble catalogue.csv et son empreinte dans manifest.json.

## Périmètre

Ce notebook complète des exports J15b déjà générés et conformes. Il ne demande
pas les grands profils bruts. Le module app/catalogue_fr.py du correctif doit
être copié dans le projet. Il couvre les 1 070 noms distincts des deux profils
communaux fournis ; les autres noms restent consultables et sont signalés à relire.

Les libellés et unités sont des propositions éditoriales. Un dictionnaire existant,
une correction utilisateur ou une ligne VALIDE=OUI est conservé en priorité.
Une unité ambiguë reste « à vérifier » ou « à préciser » ; aucune ligne n'est
validée automatiquement. Les taux de pauvreté sans seuil documenté restent à vérifier.
Le revenu médian générique n'est pas assimilé automatiquement au niveau de vie.

Enregistrer ce notebook dans notebooks/15b_clarifier_catalogue_unites_libelles.ipynb.


## 1 — Imports

Exécuter dans le même environnement que le notebook J15b complet.


In [1]:
import os
for variable in ['OMP_NUM_THREADS','NUMEXPR_NUM_THREADS','MKL_NUM_THREADS']:
    os.environ[variable]='1'
from pathlib import Path
from datetime import datetime,timezone
import sys,json,hashlib,shutil,copy
import pandas as pd
from IPython.display import display
pd.set_option('display.max_colwidth',120)


## 2 — Choisir la racine et les exports locaux

Renseigner RACINE_MANUELLE si nécessaire. Le dossier local par défaut est
data/app/j15b. Si les exports locaux ont aussi été placés dans app/data/app/j15b,
modifier uniquement DOSSIER_EXPORT_MANUEL. Le manifeste J15a doit être dans le
dossier j15 voisin du dossier j15b choisi. Les chemins effectifs sont affichés.


In [2]:
RACINE_MANUELLE=None
DOSSIER_EXPORT_MANUEL=None
# Exemple : RACINE_MANUELLE=Path(r'C:/Users/almou/OneDrive/GeoMarketing_IDF')
# Exemple : DOSSIER_EXPORT_MANUEL=RACINE_MANUELLE/'app/data/app/j15b'
racines=[Path(os.environ['GEOMARKETING_RACINE'])] if os.environ.get('GEOMARKETING_RACINE') else [Path.cwd(),*Path.cwd().parents,Path.home()/'OneDrive/GeoMarketing_IDF']
RACINE=Path(RACINE_MANUELLE) if RACINE_MANUELLE is not None else next((p for p in racines if (p/'app/catalogue_fr.py').is_file()),None)
if RACINE is None:raise FileNotFoundError('Renseigner RACINE_MANUELLE et copier le nouveau module app/catalogue_fr.py depuis le correctif.')
RACINE=RACINE.resolve()
DOSSIER_EXPORT=Path(DOSSIER_EXPORT_MANUEL) if DOSSIER_EXPORT_MANUEL is not None else RACINE/'data/app/j15b'
DOSSIER_BASE=DOSSIER_EXPORT.parent/'j15'
DOSSIER_INTERIM=RACINE/'data/interim/j15b/libelles_unites'
DOSSIER_INTERIM.mkdir(parents=True,exist_ok=True)
if str(RACINE) not in sys.path:sys.path.insert(0,str(RACINE))
from app.catalogue_fr import enrich_catalogue
from app.indicators import digest,load_catalogue,load_theme,read_table
print('Racine :',RACINE)
print('Exports J15a :',DOSSIER_BASE)
print('Exports J15b à actualiser :',DOSSIER_EXPORT)


Racine : C:\Users\almou\OneDrive\GeoMarketing_IDF
Exports J15a : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\app\j15
Exports J15b à actualiser : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\app\j15b


## 3 — Vérifier les exports avant modification

Ce contrôle doit réussir avant de modifier le catalogue. Une incohérence locale
demande de régénérer J15b à partir du socle J15a choisi. Ce notebook conserve
les contrôles d'intégrité ; il ne contourne pas une erreur de synchronisation.


In [3]:
catalogue_avant,manifest_avant=load_catalogue(DOSSIER_EXPORT,DOSSIER_BASE)
for entree in manifest_avant['fichiers']:
    p=DOSSIER_EXPORT/entree['nom']
    if not p.is_file() or digest(p)!=entree['sha256']:
        raise ValueError('Export local absent ou différent du manifeste : '+str(p))
empreintes_avant={e['nom']:e['sha256'] for e in manifest_avant['fichiers']}
print('Exports locaux conformes :',len(catalogue_avant),'variables.')


Exports locaux conformes : 1348 variables.


## 4 — Préparer et examiner les propositions

Les colonnes COLONNE et ID restent les clés techniques. Les noms lisibles sont
écrits dans LIBELLE ; les unités sont dans UNITE. Un indice de référence 100
est distingué d'un score sur 100, et un écart de taux reste en points de pourcentage.
Les observations de la BPE ne sont pas assimilées automatiquement à des sites distincts.


In [4]:
FICHIER_CORRECTIONS=RACINE/'docs/corrections_indicateurs_j15b.csv'
corrections=read_table(FICHIER_CORRECTIONS) if FICHIER_CORRECTIONS.is_file() else None
catalogue_apres,journal=enrich_catalogue(catalogue_avant,corrections=corrections)
modifie=journal['LIBELLE_AVANT'].ne(journal['LIBELLE_APRES'])|journal['UNITE_AVANT'].ne(journal['UNITE_APRES'])
display(journal.loc[modifie,['NIVEAU','COLONNE','LIBELLE_AVANT','LIBELLE_APRES','UNITE_AVANT','UNITE_APRES']].head(40))
print('Lignes actualisées :',int(modifie.sum()))
print('Lignes avec une proposition disponible :',int(journal['PROPOSITION_DISPONIBLE'].sum()))
print('Lignes déjà validées et conservées :',int(journal['METADONNEES_PROTEGEES'].sum()))
assert catalogue_apres['ID'].equals(catalogue_avant['ID'])
assert catalogue_apres['COLONNE'].equals(catalogue_avant['COLONNE'])
assert catalogue_apres['VALIDE'].equals(catalogue_avant['VALIDE'])


,NIVEAU,COLONNE,LIBELLE_AVANT,LIBELLE_APRES,UNITE_AVANT,UNITE_APRES
0,IRIS,CODGEO_REFERENCE,Codgeo reference,Codgeo reference,,Unité à préciser
1,IRIS,CODGEO_SOURCE,Codgeo source,Codgeo source,,Unité à préciser
2,IRIS,CODGEO_ETUDE,Codgeo etude,Codgeo etude,,Unité à préciser
3,IRIS,DEPARTEMENT,Departement,Departement,,Unité à préciser
4,IRIS,NOM_IRIS,Nom iris,Nom du quartier statistique,,texte
5,IRIS,NOM_COMMUNE_SOURCE,Nom commune source,Nom commune source,,Unité à préciser
6,IRIS,TYP_IRIS,Typ iris,Typ iris,,Unité à préciser
7,IRIS,NOM_COMMUNE_REFERENCE,Nom commune reference,Nom commune reference,,Unité à préciser
8,IRIS,ORIGINE_SELECTION_J12,Origine selection j12,Origine selection j12,,Unité à préciser
9,IRIS,NOM_COMMUNE_ETUDE,Nom commune etude,Nom commune etude,,Unité à préciser


Lignes actualisées : 1250
Lignes avec une proposition disponible : 1099
Lignes déjà validées et conservées : 0


## 5 — Exporter la relecture éditoriale

Ces fichiers de contrôle vont dans interim. Les propositions ne constituent
pas une validation scientifique. Relire les dénominateurs, sources et années
avant de passer un indicateur à VALIDE=OUI dans le fichier de corrections habituel.


In [ ]:
journal.to_csv(DOSSIER_INTERIM/'journal_libelles_unites_j15b.csv',index=False,sep=';',encoding='utf-8-sig')
a_revoir=catalogue_apres.loc[catalogue_apres['UNITE'].str.contains(r'à préciser|à vérifier',case=False,regex=True)|~journal['PROPOSITION_DISPONIBLE'].to_numpy()]
a_revoir.to_csv(DOSSIER_INTERIM/'variables_a_documenter_j15b.csv',index=False,sep=';',encoding='utf-8-sig')
catalogue_apres[['ID','NIVEAU','COLONNE','LIBELLE','UNITE','VALIDE','ORIGINE_LIBELLE','ORIGINE_UNITE','STATUT_UNITE']].to_csv(
    DOSSIER_INTERIM/'catalogue_libelles_unites_a_relire.csv',index=False,sep=';',encoding='utf-8-sig')
print('Variables à documenter ou à confirmer :',len(a_revoir))
print('Dossier de relecture :',DOSSIER_INTERIM)


## 6 — Sauvegarder puis actualiser catalogue et manifeste

La cellule sauvegarde les deux fichiers d'origine, écrit le nouveau catalogue,
puis met à jour seulement son empreinte et sa taille dans le manifeste J15b.
Les CSV contenant les valeurs thématiques restent identiques. Les noms,
unités et définitions éditoriales sont les seules données du catalogue actualisées.


In [ ]:
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
SAUVEGARDE=DOSSIER_INTERIM/'sauvegardes'/stamp
SAUVEGARDE.mkdir(parents=True,exist_ok=False)
for nom in ['catalogue.csv','manifest.json']:shutil.copy2(DOSSIER_EXPORT/nom,SAUVEGARDE/nom)
manifest_apres=copy.deepcopy(manifest_avant)
manifest_apres['revision_editoriale']={'date_utc':datetime.now(timezone.utc).isoformat(),
    'type':'LIBELLES_FRANCAIS_ET_UNITES','nb_lignes_actualisees':int(modifie.sum()),
    'module_sha256':digest(RACINE/'app/catalogue_fr.py'),'validation_automatique':False}
try:
    catalogue_apres.to_csv(DOSSIER_EXPORT/'catalogue.csv',index=False,encoding='utf-8-sig',lineterminator='\n')
    for entree in manifest_apres['fichiers']:
        if entree['nom']=='catalogue.csv':
            entree['sha256']=digest(DOSSIER_EXPORT/'catalogue.csv')
            entree['octets']=(DOSSIER_EXPORT/'catalogue.csv').stat().st_size
    (DOSSIER_EXPORT/'manifest.json').write_text(json.dumps(manifest_apres,ensure_ascii=False,indent=2),encoding='utf-8')
    catalogue_relu,manifest_relu=load_catalogue(DOSSIER_EXPORT,DOSSIER_BASE)
except Exception:
    for nom in ['catalogue.csv','manifest.json']:shutil.copy2(SAUVEGARDE/nom,DOSSIER_EXPORT/nom)
    raise
print('Sauvegarde :',SAUVEGARDE)
print('Catalogue et manifeste actualisés ensemble ✅')


## 7 — Vérifier les valeurs et tous les fichiers

Le contrôle couvre tous les fichiers référencés, pas seulement le catalogue.
L'empreinte du manifeste J15a reste celle de l'ensemble choisi au départ.


In [ ]:
for entree in manifest_relu['fichiers']:
    assert digest(DOSSIER_EXPORT/entree['nom'])==entree['sha256'],entree['nom']
    if entree['nom']!='catalogue.csv':assert entree['sha256']==empreintes_avant[entree['nom']],entree['nom']
for entree in manifest_relu['themes']:
    load_theme(DOSSIER_EXPORT,manifest_relu,entree['niveau'],entree['theme'])
assert manifest_relu['base_manifest_sha256']==manifest_avant['base_manifest_sha256']
assert catalogue_relu['ID'].tolist()==catalogue_avant['ID'].tolist()
assert catalogue_relu['COLONNE'].tolist()==catalogue_avant['COLONNE'].tolist()
assert catalogue_relu['VALIDE'].tolist()==catalogue_avant['VALIDE'].tolist()
print('Contrôles réussis : identifiants, valeurs thématiques, validations et empreintes conservés ✅')


## 8 — Publier dans GitHub

Publier les fichiers app/catalogue_fr.py, app/indicators.py et app/explorer.py du
correctif, puis le catalogue.csv et le manifest.json du dossier vérifié ensemble.
Avec l'organisation actuelle du déploiement, la destination est
app/data/app/j15b/ dans GitHub. Les autres CSV thématiques y restent présents.
Le fichier de référence éditoriale sous app/assets sert à la relecture.

Les versions des manifestes restent J15a = 2 et J15b = 1. Conserver les chemins
personnalisés de streamlit_app.py. Après commit, redémarrer Streamlit et vérifier
un effectif, un pourcentage, une densité, un revenu et un écart en points.

Pour les prochaines régénérations, le notebook J15b complet actualisé utilise
automatiquement les mêmes règles avant l'export. Le fichier
docs/corrections_indicateurs_j15b.csv conserve la priorité pour vos formulations
et unités vérifiées. Publier ses exports générés avec leur manifeste associé.
